# 28 · CXR-LT 2024 — kiểm kê (Stage 0)

Bước đầu của việc **làm lại từ đầu trên CXR-LT**. Notebook này không train gì cả — nó chỉ dựng
bảng chỉ mục chuẩn và trả lời các câu hỏi chặn trước khi tiêu tiền BigQuery/GPU.

Nguồn duy nhất: **`cxr-lt-2024/labels.csv`** trong file zip PhysioNet. File này đã có sẵn cột
`split` cho cả ba tập nên không phải ghép 7 file rời.

Đọc **thẳng từ trong zip** — không cần giải nén. Zip chỉ chứa 18 file CSV nên `zipfile` mở trực
tiếp nhanh hơn nhiều so với việc giải nén ra Drive.

Đầu ra:

| file | nội dung |
|---|---|
| `cxrlt_2024/cxrlt_index.parquet` | chỉ mục mức ảnh: `dicom_id, study_id, subject_id, split, view_group, fpath, has_shard` + 45 nhãn |
| `cxrlt_2024/cxrlt_inventory.json` | thống kê nhãn, phân tầng, độ phủ shard |

**CPU, ~3–6 phút.** RAM đỉnh ~2 GB (labels.csv 96 MB nở ra khi vào pandas).

## C0 · Config + định vị nguồn

In [1]:
import os, io, json, zipfile, hashlib
import numpy as np, pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT     = '/content/drive/MyDrive/MIMIC_CXR'
    DATA_DIR = os.path.join(ROOT, 'planB_out')
except Exception:
    ROOT, DATA_DIR = '.', './planB_out'

OUT_DIR   = os.path.join(DATA_DIR, 'cxrlt_2024')      # moi thu cua nhanh CXR-LT nam o day
SHARD_DIR = os.path.join(DATA_DIR, 'cxr_560_shards')  # shard anh da co (tu 04a)
os.makedirs(OUT_DIR, exist_ok=True)

ZIP  = os.path.join(ROOT, 'cxr-lt-multi-label-long-tailed-classification-on-chest-x-rays-2.0.0.zip')
BASE = 'cxr-lt-multi-label-long-tailed-classification-on-chest-x-rays-2.0.0/cxr-lt-2024'
DEST = ZIP[:-4]                                        # thu muc giai nen (neu da giai nen)

# --- doc tu thu muc da giai nen neu co, khong thi doc thang trong zip ---
def _open(name):
    p = os.path.join(DEST, BASE, name)
    if os.path.exists(p):
        return p
    assert os.path.exists(ZIP), (
        f'khong thay ca thu muc giai nen lan file zip.\n'
        f'  da tim: {p}\n'
        f'          {ZIP}')
    return zipfile.ZipFile(ZIP).open(f'{BASE}/{name}')

SRC_KIND = 'thu muc da giai nen' if os.path.exists(os.path.join(DEST, BASE)) else 'doc thang trong zip'
print(f'nguon  : {SRC_KIND}')
print(f'out    : {OUT_DIR}')
print(f'shards : {SHARD_DIR}  ({"co" if os.path.isdir(SHARD_DIR) else "KHONG THAY"})')

# nguong phan tang long-tail — theo SO CA DUONG tren train, khong theo ty le
# (voi 258k anh thi moi nhan deu co prevalence nho, chia theo ty le se don het vao mot ro)
TIER_EDGES = (1000, 10000)     # tail < 1000 <= medium < 10000 <= head
ID_COLS = ['dicom_id', 'subject_id', 'study_id', 'ViewPosition',
           'ViewCodeSequence_CodeMeaning', 'fpath', 'split']

Mounted at /content/drive
nguon  : thu muc da giai nen
out    : /content/drive/MyDrive/MIMIC_CXR/planB_out/cxrlt_2024
shards : /content/drive/MyDrive/MIMIC_CXR/planB_out/cxr_560_shards  (co)


## C1 · Nạp `labels.csv` + kiểm tra toàn vẹn

In [2]:
L = pd.read_csv(_open('labels.csv'))
LAB45 = [c for c in L.columns if c not in ID_COLS]
print(f'{len(L):,} anh x {len(L.columns)} cot | {len(LAB45)} nhan')

# --- toan ven co ban ---
assert L.dicom_id.is_unique, 'dicom_id KHONG duy nhat -> khong dung lam khoa anh duoc'
assert set(L.split) == {'train', 'val', 'test'}, f'split la {set(L.split)}'
_bad = L[LAB45].isna().sum().sum()
assert _bad == 0, f'{_bad} o nhan bi trong -> gia dinh "nhi phan hoan toan" SAI'
_vals = set()
for c in LAB45: _vals |= set(pd.unique(L[c]))
assert _vals <= {0, 1}, f'nhan co gia tri ngoai {{0,1}}: {sorted(_vals - {0,1})}'
print(f'nhan: nhi phan hoan toan, khong o rong, khong gia tri -1')

# --- CHUAN HOA ID -----------------------------------------------------------
# CXR-LT tron hai quy uoc: subject_id da la int64 (10000032) nhung study_id la
# CHUOI co tien to 's' (s50414267). MIMIC-IV tren BigQuery dung so nguyen cho ca hai,
# nen phai cat tien to NGAY O DAY — neu de den luc join BigQuery moi phat hien thi
# truy van se khop 0 dong ma khong bao loi.
_s = L.study_id.astype(str)
assert (_s.str[0] == 's').all(),      f'study_id co tien to la {sorted(set(_s.str[0]))}, khong phai "s"'
assert _s.str[1:].str.isdigit().all(), 'study_id co ky tu khong phai chu so sau tien to'
L['study_id'] = _s.str[1:].astype('int64')
assert pd.api.types.is_integer_dtype(L.subject_id), f'subject_id dtype {L.subject_id.dtype}'
print(f'study_id : chuoi "s########" -> int64  (vd {_s.iloc[0]} -> {L.study_id.iloc[0]})')
print(f'subject_id: da la int64, giu nguyen  (vd {L.subject_id.iloc[0]})')

# fpath phai khop ID -> chung minh viec cat tien to khong lam lech hang nao
_p = L.fpath.str.extract(r'files/p\d\d/p(\d+)/s(\d+)/([0-9a-f-]+)\.jpg')
assert _p.notna().all().all(), 'fpath khong dung dinh dang files/pXX/pNNN/sNNN/<dicom>.jpg'
assert (_p[0].astype('int64') == L.subject_id).all(), 'fpath lech subject_id'
assert (_p[1].astype('int64') == L.study_id).all(),   'fpath lech study_id'
assert (_p[2] == L.dicom_id).all(),                   'fpath lech dicom_id'
print('fpath     : khop ca 3 ID -> viec cat tien to khong lam lech dong nao')

print(f'\nsplit  : {L.split.value_counts().reindex(["train","val","test"]).to_dict()}')
print(f'study  : {L.study_id.nunique():,}')
print(f'subject: {L.subject_id.nunique():,}')
print(f'anh/study: trung binh {len(L)/L.study_id.nunique():.2f}')

377,110 anh x 52 cot | 45 nhan
nhan: nhi phan hoan toan, khong o rong, khong gia tri -1
study_id : chuoi "s########" -> int64  (vd s50414267 -> 50414267)
subject_id: da la int64, giu nguyen  (vd 10000032)
fpath     : khop ca 3 ID -> viec cat tien to khong lam lech dong nao

split  : {'train': 258871, 'val': 39293, 'test': 78946}
study  : 227,835
subject: 65,379
anh/study: trung binh 1.66


## C2 · Tách bộ nhãn theo task

`labels.csv` gộp cả 45 nhãn. Task 1 (mục tiêu chính) chỉ dùng 40; 5 nhãn còn lại là task 3
(zero-shot) — giữ trong chỉ mục để dành, nhưng **không** đưa vào train của task 1.

In [3]:
# lay danh sach nhan cua tung task tu chinh header cua file task do (nrows=0 -> chi doc header)
T = {}
for t in (1, 2, 3):
    cols = pd.read_csv(_open(f'development_labeled_task{t}.csv'), nrows=0).columns
    T[t] = [c for c in cols if c not in ID_COLS]
    print(f'task{t}: {len(T[t])} nhan')

LAB40, LAB_ZS = T[1], T[3]
assert len(LAB40) == 40 and len(LAB_ZS) == 5
assert set(T[2]) <= set(LAB40),              'task2 khong phai tap con cua task1'
assert not (set(LAB40) & set(LAB_ZS)),       'task1 va task3 chong nhau'
assert set(LAB40) | set(LAB_ZS) == set(LAB45), 'task1 + task3 != 45 nhan cua labels.csv'
print(f'\ntask1 (40) + task3 (5) = {len(LAB45)} nhan cua labels.csv  [khop]')
print(f'task3 zero-shot: {LAB_ZS}')

task1: 40 nhan
task2: 26 nhan
task3: 5 nhan

task1 (40) + task3 (5) = 45 nhan cua labels.csv  [khop]
task3 zero-shot: ['Bulla', 'Cardiomyopathy', 'Hilum', 'Osteopenia', 'Scoliosis']


## C3 · Thống kê nhãn + phân tầng long-tail

In [4]:
tr = L.split == 'train'
S = pd.DataFrame({'label': LAB40})
for sp in ('train', 'val', 'test'):
    S[f'n_{sp}'] = [int(L.loc[L.split == sp, c].sum()) for c in LAB40]
S['prev_train'] = S.n_train / int(tr.sum())
S['tier'] = np.where(S.n_train < TIER_EDGES[0], 'tail',
             np.where(S.n_train < TIER_EDGES[1], 'medium', 'head'))
S = S.sort_values('n_train', ascending=False).reset_index(drop=True)

print('=== 40 NHAN TASK-1, sap theo so ca duong tren TRAIN ===')
print(S.to_string(index=False, float_format=lambda v: f'{v:.4f}'))

print(f'\n=== PHAN TANG (nguong n_train: <{TIER_EDGES[0]} tail, <{TIER_EDGES[1]} medium) ===')
g = (S.groupby('tier').agg(n_nhan=('label', 'size'), pos_min=('n_train', 'min'),
                           pos_max=('n_train', 'max'), pos_tong=('n_train', 'sum'))
       .reindex(['head', 'medium', 'tail']).dropna(how='all'))
print(g.to_string())
print(f'\nmat can bang: {S.n_train.iloc[0]:,} / {S.n_train.iloc[-1]:,} = '
      f'{S.n_train.iloc[0]/max(S.n_train.iloc[-1],1):,.0f}x')

# --- canh bao phuong sai: nhan qua hiem tren VAL thi moi so AP deu khong doc duoc ---
_thin = S[S.n_val < 50]
if len(_thin):
    print(f'\n!! {len(_thin)} nhan co < 50 ca duong tren VAL — AP se rat nhieu, '
          f'BAT BUOC in kem n_pos:')
    print(_thin[['label', 'n_train', 'n_val', 'n_test']].to_string(index=False))

# --- so nhan moi anh ---
k = L[LAB40].sum(1)
print(f'\nso nhan/anh: trung binh {k.mean():.2f} | trung vi {int(k.median())} | max {int(k.max())}'
      f' | khong nhan nao {int((k==0).sum()):,} ({(k==0).mean():.2%})')

=== 40 NHAN TASK-1, sap theo so ca duong tren TRAIN ===
                     label  n_train  n_val  n_test  prev_train   tier
           Support Devices    86079  13161   27536      0.3325   head
              Lung Opacity    77482  11731   24843      0.2993   head
              Cardiomegaly    74738  11583   22976      0.2887   head
          Pleural Effusion    66401  10430   21876      0.2565   head
               Atelectasis    65376  10131   20804      0.2525   head
                 Pneumonia    46660   7061   14606      0.1802   head
                     Edema    37256   5946   11937      0.1439   head
                    Normal    34292   5088    9720      0.1325   head
Enlarged Cardiomediastinum    29628   4244    9088      0.1145   head
             Consolidation    15371   2379    5013      0.0594   head
              Pneumothorax    13858   2342    4929      0.0535   head
                  Fracture    11568   1576    3849      0.0447   head
              Infiltration    1008

## C4 · Ba cổng toàn vẹn

Ba giả định mà mọi thứ phía sau dựa vào. Kiểm ngay ở đây để nếu sai thì biết ngay, chứ không phát
hiện sau khi đã train xong.

In [5]:
GATE = {}

# --- cong 1: benh nhan khong ro ri giua cac split ---
_g = L.groupby('subject_id').split.nunique()
GATE['subject_da_split'] = int((_g > 1).sum())
_gs = L.groupby('study_id').split.nunique()
GATE['study_da_split'] = int((_gs > 1).sum())
print(f'cong 1 — subject nam o >1 split : {GATE["subject_da_split"]} / {len(_g):,}')
print(f'         study   nam o >1 split : {GATE["study_da_split"]} / {len(_gs):,}')
assert GATE['subject_da_split'] == 0 and GATE['study_da_split'] == 0, \
    'CO RO RI giua cac split -> khong duoc dung split chinh thuc nhu dang co'

# --- cong 2: nhan hang so trong cung mot study ---
# neu dung -> duoc phep lam viec o MUC STUDY va view pooling 0.7/0.3 co nghia
_v = L.groupby('study_id')[LAB45].nunique().max(axis=1)
GATE['study_nhan_khong_dong_nhat'] = int((_v > 1).sum())
print(f'\ncong 2 — study co anh khac nhan nhau: {GATE["study_nhan_khong_dong_nhat"]} / {len(_v):,}')
assert GATE['study_nhan_khong_dong_nhat'] == 0, \
    'nhan KHONG hang so trong study -> bat buoc lam o muc ANH, view pooling mat nghia'
print('         -> duoc phep lam o MUC STUDY; view pooling 0.7/0.3 hop le')

# --- cong 3: view ---
print(f'\ncong 3 — ViewPosition:')
_vp = L.ViewPosition.value_counts(dropna=False)
print(_vp.head(10).to_string())
GATE['view_thieu'] = int(L.ViewPosition.isna().sum())
print(f'         thieu ViewPosition: {GATE["view_thieu"]:,} ({GATE["view_thieu"]/len(L):.2%})'
      f'  -> can quy tac du phong (xem 03_view_resolution.ipynb)')

cong 1 — subject nam o >1 split : 0 / 65,379
         study   nam o >1 split : 0 / 227,835

cong 2 — study co anh khac nhan nhau: 0 / 227,835
         -> duoc phep lam o MUC STUDY; view pooling 0.7/0.3 hop le

cong 3 — ViewPosition:
ViewPosition
AP          147173
PA           96161
LATERAL      82853
LL           35133
NaN          15769
PA LLD           4
LAO              3
RAO              3
AP AXIAL         2
AP LLD           2
         thieu ViewPosition: 15,769 (4.18%)  -> can quy tac du phong (xem 03_view_resolution.ipynb)


## C5 · Gán `view_group` (frontal / lateral / unknown)

Cùng quy ước với `03_view_resolution.ipynb`: gộp về hai nhóm để dùng được view pooling
`0.7·frontal + 0.3·lateral`. Ô trống `ViewPosition` để `unknown` — C6 báo cáo số lượng, và
`29b`/`31` sẽ quyết định xử lý (chưa vứt ở đây).

In [6]:
FRONTAL = {'AP', 'PA', 'AP AXIAL', 'PA LLD', 'PA RLD', 'AP LLD', 'AP RLD'}
LATERAL = {'LATERAL', 'LL', 'RL', 'LAO', 'RAO', 'LPO', 'RPO', 'SWIMMERS', 'XTABLE LATERAL'}

_vp = L.ViewPosition.fillna('').str.upper().str.strip()
L['view_group'] = np.select([_vp.isin(FRONTAL), _vp.isin(LATERAL)],
                            ['frontal', 'lateral'], default='unknown')
print(L.view_group.value_counts().to_frame('n')
        .assign(ty_le=lambda d: (d.n / len(L)).map('{:.2%}'.format)).to_string())

# nhung gia tri ViewPosition KHONG duoc phan loai — de biet co bo sot loai view nao khong
_un = L.loc[L.view_group == 'unknown', 'ViewPosition'].value_counts(dropna=False)
print(f'\ngia tri chua phan loai duoc ({len(_un)} loai):')
print(_un.head(12).to_string())

# co bao nhieu study khong con anh frontal nao
_nf = L[L.view_group == 'frontal'].groupby('study_id').size()
_no_front = L.study_id.nunique() - len(_nf)
print(f'\nstudy KHONG co anh frontal nao: {_no_front:,} / {L.study_id.nunique():,} '
      f'({_no_front/L.study_id.nunique():.2%})')

                 n   ty_le
view_group                
frontal     243345  64.53%
lateral     117996  31.29%
unknown      15769   4.18%

gia tri chua phan loai duoc (1 loai):
ViewPosition
NaN    15769

study KHONG co anh frontal nao: 9,695 / 227,835 (4.26%)


## C6 · Độ phủ shard ảnh 560px

`04a_preprocess_images.ipynb` ghi `_work_list.parquet` vào `cxr_560_shards/` — đó là danh sách
chính xác những `dicom_id` đã được đưa vào shard. Dùng file đó thay vì quét ~400 file `.tar`
(nhanh hơn hàng chục lần trên Drive).

Phần thiếu **sẽ kéo về từ MIMIC-CXR-JPG** — đây là số liệu để biết cần kéo bao nhiêu, không phải
lý do để thu hẹp cohort.

In [7]:
WL = os.path.join(SHARD_DIR, '_work_list.parquet')
if os.path.exists(WL):
    have = set(pd.read_parquet(WL, columns=['dicom_id']).dicom_id)
    L['has_shard'] = L.dicom_id.isin(have)
    n_have = int(L.has_shard.sum())
    print(f'shard hien co    : {len(have):,} anh')
    print(f'CXR-LT da co     : {n_have:,} / {len(L):,}  ({n_have/len(L):.2%})')
    print(f'CAN KEO THEM     : {len(L)-n_have:,} anh')
    print('\nthieu, tach theo split:')
    print((L.groupby('split').has_shard.agg(tong='size', da_co='sum')
             .assign(thieu=lambda d: d.tong - d.da_co,
                     ty_le=lambda d: (d.da_co / d.tong).map('{:.2%}'.format))
             .reindex(['train', 'val', 'test'])).to_string())
    # bao nhieu anh trong shard KHONG thuoc CXR-LT -> phan se khong dung toi
    print(f'\nanh trong shard nhung khong thuoc CXR-LT: {len(have - set(L.dicom_id)):,}')
else:
    L['has_shard'] = False
    print(f'!! khong thay {WL}')
    print('   -> chua danh gia duoc do phu; coi nhu phai keo toan bo 377,110 anh.')

shard hien co    : 346,578 anh
CXR-LT da co     : 346,578 / 377,110  (91.90%)
CAN KEO THEM     : 30,532 anh

thieu, tach theo split:
         tong   da_co  thieu   ty_le
split                               
train  258871  238208  20663  92.02%
val     39293   36012   3281  91.65%
test    78946   72358   6588  91.66%

anh trong shard nhung khong thuoc CXR-LT: 0


## C7 · Ghi chỉ mục + `cxrlt_inventory.json`

In [8]:
IDX_COLS = ['dicom_id', 'study_id', 'subject_id', 'split', 'view_group',
            'ViewPosition', 'fpath', 'has_shard']
IDX = L[IDX_COLS + LAB45].copy()
assert IDX.study_id.dtype == 'int64' and IDX.subject_id.dtype == 'int64'   # da chuan hoa o C1
for c in LAB45: IDX[c] = IDX[c].astype('int8')

P_IDX = os.path.join(OUT_DIR, 'cxrlt_index.parquet')
IDX.to_parquet(P_IDX, index=False)
_chk = pd.read_parquet(P_IDX)
assert len(_chk) == len(L) and _chk.dicom_id.is_unique
print(f'ghi {P_IDX}  ({os.path.getsize(P_IDX)/1e6:.1f} MB, {len(_chk):,} dong)')

INV = dict(
    nguon='cxr-lt-2024/labels.csv (PhysioNet cxr-lt 2.0.0)',
    n_anh=len(L), n_study=int(L.study_id.nunique()), n_subject=int(L.subject_id.nunique()),
    split={k: int(v) for k, v in L.split.value_counts().items()},
    labels_task1=LAB40, labels_zeroshot_task3=LAB_ZS,
    tier_edges=list(TIER_EDGES),
    tier={t: S.loc[S.tier == t, 'label'].tolist() for t in ('head', 'medium', 'tail')},
    n_pos_train={r.label: int(r.n_train) for r in S.itertuples()},
    n_pos_val={r.label: int(r.n_val) for r in S.itertuples()},
    gate=GATE,
    view_group={k: int(v) for k, v in L.view_group.value_counts().items()},
    shard_da_co=int(L.has_shard.sum()), shard_can_keo=int((~L.has_shard).sum()),
)
P_INV = os.path.join(OUT_DIR, 'cxrlt_inventory.json')
json.dump(INV, open(P_INV, 'w'), indent=1)
print(f'ghi {P_INV}')

print(f'''
================================================================
STAGE 0 XONG

  anh {INV['n_anh']:,} | study {INV['n_study']:,} | subject {INV['n_subject']:,}
  40 nhan task-1: head {len(INV['tier']['head'])} / medium {len(INV['tier']['medium'])}
                  / tail {len(INV['tier']['tail'])}
  shard: da co {INV['shard_da_co']:,} | can keo {INV['shard_can_keo']:,}

  Ba cong toan ven: DAT (khong ro ri split, nhan hang so trong study, dicom_id duy nhat)

TIEP THEO -> 29_cxrlt_ehr_base.ipynb
  dung index_time + cur_hadm/cur_edstay/setting cho {INV['n_study']:,} study.
  Do phu ENCOUNTER se duoc bao cao o do (cung mot truy van, khong chay hai lan).
================================================================''')

ghi /content/drive/MyDrive/MIMIC_CXR/planB_out/cxrlt_2024/cxrlt_index.parquet  (40.3 MB, 377,110 dong)
ghi /content/drive/MyDrive/MIMIC_CXR/planB_out/cxrlt_2024/cxrlt_inventory.json

STAGE 0 XONG

  anh 377,110 | study 227,835 | subject 65,379
  40 nhan task-1: head 13 / medium 16
                  / tail 11
  shard: da co 346,578 | can keo 30,532

  Ba cong toan ven: DAT (khong ro ri split, nhan hang so trong study, dicom_id duy nhat)

TIEP THEO -> 29_cxrlt_ehr_base.ipynb
  dung index_time + cur_hadm/cur_edstay/setting cho 227,835 study.
  Do phu ENCOUNTER se duoc bao cao o do (cung mot truy van, khong chay hai lan).
